# Import library


In [ ]:
import os
import pickle
import sys

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler

sys.path.append(os.path.abspath(os.path.join(os.path.dirname("pipoint"), "..")))
from torch.utils.data import DataLoader

from utils.dataset import CachedPIRawBatchDataset
from utils.evalmetrics import EvalDFBuilder, EvaluationMetrics

# Import data


In [ ]:
datafolderpath = "../data"
scaler_path = "../save_model/scaler"
pretrained_model_dir = "../save_model/lstm_encdec"  # path to the pretrained model directory

In [ ]:
dict_path = os.path.join(datafolderpath, "sample_data_paths_16_numlags_192_df_file.pkl")
with open(dict_path, "rb") as pickle_file:
    data = pickle.load(pickle_file)
print(data.keys())

target_cols = data["target_col"]
features_list = data["features_list"]
future_regressor = data["future_regressor"]
lag_regressor = data["lag_regressor"]
num_step_ahead = data["num_step_ahead"]
data_resolution = data["resolution"]

df_test: pd.DataFrame = pd.read_csv("../" + data["df_test_nonan"])  # pd.DataFrame
df_test.drop(columns="Datetime", inplace=True)

load prefitted-scaler

In [ ]:
with open(scaler_path + "/target_scaler.pkl", "rb") as scaler_file:
    y_scaler: StandardScaler = pickle.load(scaler_file)

Define dataloader

In [ ]:
batch_size = 16
dataset = CachedPIRawBatchDataset(
    cache_dir=f"../data/raw/test/b{batch_size}",
    batch_size=batch_size,
    num_lag=16,
    shuffle=False,
    chronos_future=False,
)
loader = DataLoader(dataset=dataset, batch_size=None, pin_memory=False)

# Load Chronos-2 model


In [ ]:
from chronos import (
    BaseChronosPipeline,
    Chronos2Pipeline,
)

model_name = "amazon/chronos-2"
device = "cuda" if torch.cuda.is_available() else "cpu"

pipeline: Chronos2Pipeline = BaseChronosPipeline.from_pretrained(model_name, device_map=device)

In [ ]:
def chronos_predict(loader, batch_size=256, cross_learning=False):
    quantiles_list, point_list = [], []

    for batch in loader:
        (lag, future), _ = batch
        quantiles, point = pipeline.predict_quantiles(
            inputs=lag,
            quantile_levels=[0.05, 0.5, 0.95],
            prediction_length=num_step_ahead,
            batch_size=batch_size,
            cross_learning=cross_learning,
        )
        quantiles_list.extend(quantiles)
        point_list.extend(point)

    return quantiles_list, point_list

# Performance evaluation


## Select the dataset for evaluation


In [ ]:
metrics = EvaluationMetrics()
data_name = ["Test"]

y_test = []
for *_, target in loader:
    target = target.to("cpu").to(torch.float32)
    y_test.append(target.numpy())

y_test = np.concatenate(y_test)
y_test = y_scaler.inverse_transform(y_test)

y_eval = y_test

## Model inference


In [ ]:
quantiles, point = chronos_predict(loader, batch_size=2048, cross_learning=False)

In [ ]:
pred = torch.stack(quantiles, dim=0)  # covariate prediction samples

In [ ]:
pred.shape

In [ ]:
pred = pred[:, 0, :, :].cpu().numpy()  # only I (target variable)

In [ ]:
upper = pred[:, :, 2].astype(float)  # quantile 0.95
lower = pred[:, :, 0].astype(float)  # quantile 0.05
yhat = pred[:, :, 1].astype(float)  # quantile 0.5

In [ ]:
prediction = np.stack((lower, yhat, upper), axis=0)
os.makedirs("results", exist_ok=True)

np.save("results/chronos_zeroshot", prediction)
np.save("results/y_eval", y_eval)

## Mode1: Evaluate for entire test set 


In [ ]:
eval_all = metrics.evaluate_all(
    y=y_eval,
    upper=upper,
    lower=lower,
    yhat=yhat,
    ytarget=y_eval,
    normalize=False,
    delta=0.1,
    quantile=0.5,
)

eval_all["PINALW"] = eval_all["PINALW"].astype(float)

metrics.report_performance(eval_all, axis_mode=1)

## Mode2: Evaluate for test dataset during daytime (6:00 to 18:00)


### PI + Point forecast mode


In [ ]:
df_test_date = pd.read_csv("../" + data["df_test_date"])
df_test_date.set_index("Datetime", inplace=True)
df_test_date.index = pd.to_datetime(df_test_date.index)

In [ ]:
builder = EvalDFBuilder(num_max_workers=40, verbose=False)

df_infer = builder.build_inference_df(df_test_date, upper=upper, lower=lower, yhat=yhat)
df_eval = builder.build_evaluation_df(df_infer, upper=upper, lower=lower, yhat=yhat)

df_eval.dropna(inplace=True)

y_eval_daytime, yhat_eval_daytime, upper_eval_daytime, lower_eval_daytime = (
    builder.extract_daytime_arrays(df_eval, start_time="06:00", end_time="18:00")
)

eval_all = metrics.evaluate_all(
    y_eval_daytime,
    upper_eval_daytime,
    lower_eval_daytime,
    yhat=yhat_eval_daytime,
    ytarget=y_eval_daytime,
    normalize=False,
    delta=0.1,
    quantile=0.5,
)

eval_all["PINALW"] = eval_all["PINALW"].astype(float)

metrics.report_performance(eval_all, axis_mode=1)

### Timeseries plot


#### For PI + Point forecast


In [ ]:
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook"  # or "notebook_connected"

# Configuration
max_steps = 16
methodname = "Sum-k"

fig = go.Figure()

chunk_size = 5000  # Number of points to plot at once

# --- 1. Generate Traces for all steps ---
for step in range(1, max_steps + 1):
    idx = step - 1

    # Slice data for the current step
    y_plot = y_eval[:chunk_size, idx]
    yhat_plot = yhat[:chunk_size, idx]
    upper_plot = upper[:chunk_size, idx]
    lower_plot = lower[:chunk_size, idx]

    # Determine visibility (only Step 1 is visible initially)
    is_visible = step == 1

    fig.add_trace(
        go.Scatter(
            x=list(range(len(y_plot))),
            y=upper_plot,
            mode="lines",
            line=dict(color="red", width=1),
            name=f"{methodname} Upper",
            visible=is_visible,
            legendgroup="upper",
            showlegend=True,
        )
    )

    fig.add_trace(
        go.Scatter(
            x=list(range(len(y_plot))),
            y=lower_plot,
            mode="lines",
            line=dict(color="green", width=1),
            fill="tonexty",
            fillcolor="rgba(255, 255, 0, 0.2)",
            name=f"{methodname} Lower",
            visible=is_visible,
            legendgroup="lower",
            showlegend=True,
        )
    )

    fig.add_trace(
        go.Scatter(
            x=list(range(len(y_plot))),
            y=yhat_plot,
            mode="lines",
            line=dict(color="blue", width=2),
            name=f"{methodname} Prediction",
            visible=is_visible,
            legendgroup="pred",
            showlegend=True,
        )
    )

    # 4. True values (Black)
    fig.add_trace(
        go.Scatter(
            x=list(range(len(y_plot))),
            y=y_plot,
            mode="lines",
            line=dict(color="black", width=2),
            name="True",
            visible=is_visible,
            legendgroup="true",
            showlegend=True,
        )
    )

# --- 2. Create Slider Steps ---
slider_steps = []
traces_per_step = 4  # We added 4 traces per loop iteration above

for i in range(max_steps):
    step_val = i + 1

    step_visible = [False] * (max_steps * traces_per_step)
    start_idx = i * traces_per_step
    step_visible[start_idx : start_idx + traces_per_step] = [True] * traces_per_step

    _yhat = yhat[:, i]
    _upper = upper[:, i]
    _lower = lower[:, i]
    num_violations = ((_yhat < _lower) | (_yhat > _upper)).sum()

    step_dict = dict(
        method="update",
        args=[
            {"visible": step_visible},  # Update trace visibility
            {
                "title": f"{step_val} Step Ahead Forecast (Violations: {num_violations})"
            },  # Update Layout Title
        ],
        label=str(step_val),
    )
    slider_steps.append(step_dict)

init_violations = ((yhat[:, 0] < lower[:, 0]) | (yhat[:, 0] > upper[:, 0])).sum()

fig.update_layout(
    title=f"1 Step Ahead Forecast (Violations: {init_violations})",
    xaxis_title="Time Index",
    yaxis_title="Value",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    sliders=[
        dict(active=0, currentvalue={"prefix": "Step Ahead: "}, pad={"t": 50}, steps=slider_steps)
    ],
)

fig.show()